# Финальное сравнение всех моделей детекции фейковых новостей

К сравнению из `all_comparison.ipynb` добавлена **DeepSeek API** (zero/few-shot через REST).

**Все модели оцениваются на одном in-distribution hold-out сплите** (SEED=42, test_size=0.1, stratify=label).

**Состав:**
- Классические: LR/NB/RF + TF-IDF, LR/RF + Word2Vec
- Трансформер: RuBERT (fine-tuned)
- LLM (локальные): ruGPT-3 + LoRA V1 / V2 / V3 / V3 Tuned
- **LLM (API): DeepSeek (`deepseek-chat`) — 20-shot из train-сплита**

## 1. Импорты и настройки

In [ ]:
import os, sys, re, pickle, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from gensim.models import KeyedVectors
import joblib
import nltk
from openai import OpenAI

warnings.filterwarnings('ignore')

# Поднимаемся в корень проекта
os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'))
print('Working dir:', os.getcwd())

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)

## 2. Данные: тот же сплит, что и в all_comparison

test_size=0.1, stratify=label, SEED=42 — 441 пример, 219 правды + 222 фейка.
Все модели уже видели свои train-сплиты, эти 441 — hold-out, который никто не видел при обучении.

In [ ]:
DATA_PATH = 'data/ready_dataset.csv'
df = pd.read_csv(DATA_PATH)
df = df[['headline_clean', 'body_clean', 'combined_text', 'label']].dropna()
df['headline_clean'] = df['headline_clean'].astype(str).str.strip()
df['body_clean']     = df['body_clean'].astype(str).str.strip()
df['combined_text']  = df['combined_text'].astype(str).str.strip()
df = df[(df['headline_clean'] != '') & (df['body_clean'] != '')]
df['label'] = pd.to_numeric(df['label'], errors='coerce').astype(int)
df = df.reset_index(drop=True)

train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=SEED, stratify=df['label']
)
test_df = test_df.reset_index(drop=True)

y_true        = test_df['label'].values
headlines     = test_df['headline_clean'].astype(str).tolist()
bodies        = test_df['body_clean'].astype(str).tolist()
combined_text = test_df['combined_text'].astype(str).tolist()

print(f'Тестовая выборка: {len(test_df)} примеров (in-distribution hold-out)')
print(f'  Реальные (1): {(y_true == 1).sum()}')
print(f'  Фейки    (0): {(y_true == 0).sum()}')

# Препроцессинг для классических моделей
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('russian'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'\S+@\S+', '', t)
    t = re.sub(r'<.*?>', '', t)
    t = re.sub(r'[^а-яёa-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return ' '.join(w for w in t.split() if w not in STOPWORDS and len(w) > 2)

## 3. Загрузка моделей

In [ ]:
from peft import PeftModel
GPT_BASE = 'ai-forever/rugpt3small_based_on_gpt2'

# --- TF-IDF ---
tfidf_vec = pickle.load(open('models/tfidf_vectorizer_tf.pkl', 'rb'))
tfidf_lr  = pickle.load(open('models/logistic_regression_model_tf.pkl', 'rb'))
tfidf_nb  = pickle.load(open('models/naive_bayes_model_tf.pkl', 'rb'))
tfidf_rf  = pickle.load(open('models/random_forest_model_tf.pkl', 'rb'))
print('TF-IDF загружены')

# --- Word2Vec ---
w2v_kv = KeyedVectors.load('models/w2v_vectors.kv')
w2v_lr = pickle.load(open('models/logisticregression_model.pkl', 'rb'))
w2v_rf = pickle.load(open('models/randomforest_model.pkl', 'rb'))
print('Word2Vec загружены')

# --- RuBERT ---
rubert_tok = AutoTokenizer.from_pretrained('models/rubert/best_model')
rubert_mdl = AutoModelForSequenceClassification.from_pretrained('models/rubert/best_model').to(DEVICE)
rubert_mdl.eval()
print('RuBERT загружен')

# --- LLM V1 ---
v1_tok = AutoTokenizer.from_pretrained('models/llm/rugpt3_lora')
v1_tok.padding_side = 'left'
v1_base = AutoModelForSequenceClassification.from_pretrained(GPT_BASE, num_labels=2)
v1_base.config.pad_token_id = v1_tok.pad_token_id
v1_mdl = PeftModel.from_pretrained(v1_base, 'models/llm/rugpt3_lora').merge_and_unload().to(DEVICE)
v1_mdl.eval()
print('LLM V1 загружен')

# --- LLM V2 ---
v2_tok = AutoTokenizer.from_pretrained(GPT_BASE)
if v2_tok.pad_token is None:
    v2_tok.pad_token = v2_tok.eos_token
v2_enc = AutoModel.from_pretrained(GPT_BASE).to(DEVICE)
v2_enc.eval()
for p in v2_enc.parameters():
    p.requires_grad = False
v2_scaler = joblib.load('models/llm_v2/scaler.pkl')
v2_classifiers = joblib.load('models/llm_v2/all_classifiers.pkl')
print('LLM V2 загружен')

# --- LLM V3 ---
v3_tok = AutoTokenizer.from_pretrained('./models/llm_v3/lora_adapter')
v3_tok.padding_side = 'left'
if v3_tok.pad_token is None:
    v3_tok.pad_token = v3_tok.eos_token
v3_base = AutoModelForSequenceClassification.from_pretrained(GPT_BASE, num_labels=2)
v3_base.config.pad_token_id = v3_tok.pad_token_id
v3_mdl = PeftModel.from_pretrained(v3_base, './models/llm_v3/lora_adapter').merge_and_unload().to(DEVICE)
v3_mdl.eval()
print('LLM V3 загружен')

# --- LLM V3 Tuned ---
v3t_tok = AutoTokenizer.from_pretrained('./models/llm_v3_tuned/lora_adapter')
v3t_tok.padding_side = 'left'
if v3t_tok.pad_token is None:
    v3t_tok.pad_token = v3t_tok.eos_token
v3t_base = AutoModelForSequenceClassification.from_pretrained(GPT_BASE, num_labels=2)
v3t_base.config.pad_token_id = v3t_tok.pad_token_id
v3t_mdl = PeftModel.from_pretrained(v3t_base, './models/llm_v3_tuned/lora_adapter').merge_and_unload().to(DEVICE)
v3t_mdl.eval()
print('LLM V3 Tuned загружен')

print('\nЛокальные модели готовы')

## 4. Инференс классических моделей и RuBERT

In [ ]:
def doc_vector(tokens, kv_model):
    vecs = [kv_model[w] for w in tokens if w in kv_model]
    return np.vstack(vecs).mean(axis=0) if vecs else np.zeros(kv_model.vector_size, dtype=np.float32)

# TF-IDF
print('TF-IDF...')
h_clean = [preprocess_text(h) for h in headlines]
b_clean = [preprocess_text(b) for b in bodies]
X_tfidf = tfidf_vec.transform([f'{h} {b}' for h, b in zip(h_clean, b_clean)])
preds_tfidf_lr = tfidf_lr.predict(X_tfidf); probs_tfidf_lr = tfidf_lr.predict_proba(X_tfidf)[:, 1]
preds_tfidf_nb = tfidf_nb.predict(X_tfidf); probs_tfidf_nb = tfidf_nb.predict_proba(X_tfidf)[:, 1]
preds_tfidf_rf = tfidf_rf.predict(X_tfidf); probs_tfidf_rf = tfidf_rf.predict_proba(X_tfidf)[:, 1]

# Word2Vec
print('Word2Vec...')
w2v_feats = []
for h, b in zip(h_clean, b_clean):
    htoks = h.split()[:150]; btoks = b.split()[:150]
    h_vec = doc_vector(htoks, w2v_kv); b_vec = doc_vector(btoks, w2v_kv)
    cos_sim = float(np.dot(h_vec, b_vec) / max(np.linalg.norm(h_vec) * np.linalg.norm(b_vec), 1e-8))
    jacc = len(set(htoks) & set(btoks)) / max(1, len(set(htoks) | set(btoks)))
    ovr  = len(set(htoks) & set(btoks)) / max(1, len(set(htoks)))
    l2   = float(np.linalg.norm(h_vec - b_vec))
    feat = np.hstack([h_vec, b_vec, np.abs(h_vec - b_vec), h_vec * b_vec, [cos_sim, jacc, ovr, l2]])
    w2v_feats.append(feat)
X_w2v = np.array(w2v_feats)
preds_w2v_lr = w2v_lr.predict(X_w2v); probs_w2v_lr = w2v_lr.predict_proba(X_w2v)[:, 1]
preds_w2v_rf = w2v_rf.predict(X_w2v); probs_w2v_rf = w2v_rf.predict_proba(X_w2v)[:, 1]

# RuBERT
print('RuBERT...')
preds_rubert, probs_rubert = [], []
with torch.no_grad():
    for h, b in tqdm(zip(headlines, bodies), total=len(headlines), desc='RuBERT'):
        enc = rubert_tok(f'{h} {b}', truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        out = rubert_mdl(input_ids=enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
        p = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        preds_rubert.append(int(np.argmax(p))); probs_rubert.append(float(p[1]))
preds_rubert = np.array(preds_rubert); probs_rubert = np.array(probs_rubert)
print('Готово')

## 5. Инференс локальных LLM (V1, V2, V3, V3 Tuned)

In [ ]:
# LLM V1
print('LLM V1...')
preds_v1, probs_v1 = [], []
with torch.no_grad():
    for h, b in tqdm(zip(headlines, bodies), total=len(headlines), desc='V1'):
        enc = v1_tok(f'{h} | {b}', truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        out = v1_mdl(input_ids=enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
        p = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        preds_v1.append(int(np.argmax(p))); probs_v1.append(float(p[1]))
preds_v1 = np.array(preds_v1); probs_v1 = np.array(probs_v1)

# LLM V2
print('LLM V2...')
preds_v2, probs_v2 = [], []
with torch.inference_mode():
    for h, b in tqdm(zip(headlines, bodies), total=len(headlines), desc='V2'):
        def encode_v2(text):
            enc = v2_tok(str(text).strip(), truncation=True, max_length=64, padding=True, return_tensors='pt')
            hidden = v2_enc(input_ids=enc['input_ids'].to(DEVICE),
                            attention_mask=enc['attention_mask'].to(DEVICE)).last_hidden_state
            mf = enc['attention_mask'].to(DEVICE).unsqueeze(-1).float()
            return (hidden * mf).sum(1) / mf.sum(1).clamp(min=1e-9)
        h_emb = encode_v2(h); b_emb = encode_v2(b)
        feats = torch.cat([h_emb, b_emb, (h_emb - b_emb).abs(), h_emb * b_emb], dim=-1).cpu().numpy()
        feats_scaled = v2_scaler.transform(feats)
        prob = np.zeros(2)
        for _, clf in v2_classifiers.items():
            prob += clf.predict_proba(feats_scaled)[0]
        prob /= len(v2_classifiers)
        preds_v2.append(int(np.argmax(prob))); probs_v2.append(float(prob[1]))
preds_v2 = np.array(preds_v2); probs_v2 = np.array(probs_v2)

# LLM V3
print('LLM V3...')
preds_v3, probs_v3 = [], []
with torch.no_grad():
    for h, b in tqdm(zip(headlines, bodies), total=len(headlines), desc='V3'):
        enc = v3_tok(f'{h} | {b}', truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        out = v3_mdl(input_ids=enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
        p = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        preds_v3.append(int(np.argmax(p))); probs_v3.append(float(p[1]))
preds_v3 = np.array(preds_v3); probs_v3 = np.array(probs_v3)

# LLM V3 Tuned
print('LLM V3 Tuned...')
preds_v3t, probs_v3t = [], []
with torch.no_grad():
    for h, b in tqdm(zip(headlines, bodies), total=len(headlines), desc='V3T'):
        enc = v3t_tok(f'{h} | {b}', truncation=True, padding='max_length', max_length=512, return_tensors='pt')
        out = v3t_mdl(input_ids=enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
        p = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
        preds_v3t.append(int(np.argmax(p))); probs_v3t.append(float(p[1]))
preds_v3t = np.array(preds_v3t); probs_v3t = np.array(probs_v3t)

print('Локальные LLM готовы')

## 6. DeepSeek API: 20-shot классификация на тех же 441 примерах

Few-shot примеры берутся из **train-сплита** (тот же SEED, не пересекается с test). Single-shot, T=0, ~3-5 минут.

In [ ]:
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Retrieval-based few-shot: для каждого test-примера выберем top-K похожих train-примеров
# через TF-IDF cosine. Это превращает API-классификацию в LLM-усиленный k-NN.
retriever = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
train_texts  = train_df['combined_text'].astype(str).tolist()
train_labels = train_df['label'].astype(int).tolist()
X_train_tfidf = retriever.fit_transform(train_texts)

K_RETRIEVE = 16  # 8 ближайших фейков + 8 ближайших правды

def retrieve_fewshot(test_text, k=K_RETRIEVE):
    """top-K похожих примеров из train, сбалансированные по классам если возможно."""
    q = retriever.transform([test_text])
    sims = cosine_similarity(q, X_train_tfidf).ravel()
    order = sims.argsort()[::-1]
    target_per_class = k // 2
    by_class = {0: [], 1: []}
    for i in order:
        lbl = train_labels[i]
        if len(by_class[lbl]) < target_per_class:
            by_class[lbl].append({'text': train_texts[i], 'label': lbl, 'sim': float(sims[i])})
        if len(by_class[0]) >= target_per_class and len(by_class[1]) >= target_per_class:
            break
    examples = by_class[0] + by_class[1]
    random.Random(42).shuffle(examples)
    return examples

# Sanity check
sample_query = combined_text[0]
sample_fs = retrieve_fewshot(sample_query)
print(f'TF-IDF ретривер построен на {len(train_texts)} train-примерах')
print(f'K_RETRIEVE = {K_RETRIEVE} (по 8 на класс)')
print(f'\nДемо для test[0]:')
print(f'  query: {sample_query[:150]}...')
print(f'  получено {len(sample_fs)} примеров, sim диапазон: {min(e["sim"] for e in sample_fs):.3f} — {max(e["sim"] for e in sample_fs):.3f}')

In [ ]:
DEEPSEEK_API_KEY  = os.environ.get('DEEPSEEK_API_KEY', 'sk-28313642146841358eb4a1153feb26d6')
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_MODEL    = 'deepseek-chat'

ds_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

# Простой промпт — основная работа делается RETRIEVED few-shot примерами.
DS_SYSTEM_PROMPT = (
    'You classify Russian-language news as REAL (label=1) or FAKE (label=0). '
    'Texts are lemmatized (no punctuation, no headlines). '
    'Stylistic signals are useless — match TOPICAL patterns from the labeled examples below. '
    'You will see examples retrieved from a labeled dataset that are most similar to the query. '
    'Use them as your reference: if the query matches the topic/character of a labeled fake — answer 0; '
    'if it matches a labeled real — answer 1. '
    'Return strictly one JSON: {"label": 1 or 0, "reason": "short"}. Always return JSON.'
)

def build_messages(text, fewshot):
    msgs = [{'role': 'system', 'content': DS_SYSTEM_PROMPT}]
    for ex in fewshot:
        msgs.append({
            'role': 'user',
            'content': f'TEXT:\n{ex["text"][:1200]}\n\nClassify (return JSON):'
        })
        reason = ('similar to known fake (political satire / fabricated claim).'
                  if ex['label'] == 0 else
                  'similar to known real (factual reporting on documented events).')
        msgs.append({
            'role': 'assistant',
            'content': json.dumps({'label': ex['label'], 'reason': reason}, ensure_ascii=False)
        })
    msgs.append({
        'role': 'user',
        'content': f'TEXT:\n{text[:1200]}\n\nClassify (return JSON):'
    })
    return msgs

def parse_deepseek(content):
    content = content.strip()
    if '```' in content:
        for part in content.split('```'):
            part = part.strip()
            if part.startswith('json'):
                part = part[4:].strip()
            if part.startswith('{'):
                content = part; break
    matches = re.findall(r'\{[^{}]*\}', content, re.DOTALL)
    if matches:
        content = matches[-1]
    try:
        data = json.loads(content)
        for k in data:
            if k.lower() in ('label', 'is_true', 'is_real', 'class', 'result'):
                raw = data[k]
                if isinstance(raw, bool): return int(raw)
                if isinstance(raw, (int, float)): return int(bool(int(raw)))
                s = str(raw).strip().lower()
                return 1 if s in ('1', 'true', 'real', 'правда') else 0
    except Exception:
        pass
    m = re.findall(r'"label"\s*:\s*([01])', content.lower())
    if m: return int(m[-1])
    m = re.findall(r'"label"\s*:\s*(true|false)', content.lower())
    if m: return 1 if m[-1] == 'true' else 0
    return None

def deepseek_classify(text, max_retries=3):
    """Single-shot, T=0, retrieved few-shot."""
    fewshot = retrieve_fewshot(text)
    for attempt in range(max_retries):
        try:
            r = ds_client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                messages=build_messages(text, fewshot),
                max_tokens=128,
                temperature=0.0,
                stream=False,
            )
            label = parse_deepseek(r.choices[0].message.content.strip())
            if label is not None:
                return label
        except Exception as e:
            if attempt == max_retries - 1:
                pass
        time.sleep(0.5)
    return 1  # fallback

print('DeepSeek client готов: retrieval-based 16-shot, single-shot T=0')

In [ ]:
# Прогон DeepSeek с retrieval-based few-shot. Single-shot, T=0.
# Время: ~5 минут (441 запрос).
# Новый кэш v3 — старые v1/v2 не подхватятся.
DS_CACHE = Path('models/deepseek/predictions_441_v3.csv')

if DS_CACHE.exists():
    print(f'Найден кэш: {DS_CACHE}')
    cache_df = pd.read_csv(DS_CACHE)
    if len(cache_df) == len(test_df):
        preds_deepseek = cache_df['pred'].values.astype(int)
        print(f'Загружено {len(preds_deepseek)} предсказаний из кэша')
    else:
        print(f'Кэш ({len(cache_df)}) не совпадает с test ({len(test_df)}) — пересчитываем')
        cache_df = None
else:
    cache_df = None

if cache_df is None:
    preds_deepseek = []
    for text in tqdm(combined_text, desc='DeepSeek (retrieval 16-shot)'):
        preds_deepseek.append(deepseek_classify(text))
        time.sleep(0.15)
    preds_deepseek = np.array(preds_deepseek)
    pd.DataFrame({
        'true_label': y_true,
        'pred':       preds_deepseek,
    }).to_csv(DS_CACHE, index=False)
    print(f'Сохранено в {DS_CACHE}')

probs_deepseek = preds_deepseek.astype(float)
acc_ds = accuracy_score(y_true, preds_deepseek)
err_ds = (preds_deepseek != y_true).sum()
print(f'\nDeepSeek accuracy: {acc_ds:.4f}')
print(f'  Errors: {err_ds} / {len(y_true)} (RuBERT для сравнения: 8 ошибок = 0.9819)')
cm_ds = confusion_matrix(y_true, preds_deepseek)
print(f'  TN={cm_ds[0,0]}  FP={cm_ds[0,1]}  FN={cm_ds[1,0]}  TP={cm_ds[1,1]}')

## 7. Сводная таблица метрик

In [ ]:
all_models = {
    'LR + TF-IDF':           {'preds': preds_tfidf_lr, 'probs': probs_tfidf_lr, 'category': 'Классическая'},
    'NB + TF-IDF':           {'preds': preds_tfidf_nb, 'probs': probs_tfidf_nb, 'category': 'Классическая'},
    'RF + TF-IDF':           {'preds': preds_tfidf_rf, 'probs': probs_tfidf_rf, 'category': 'Классическая'},
    'LR + Word2Vec':         {'preds': preds_w2v_lr,   'probs': probs_w2v_lr,   'category': 'Классическая'},
    'RF + Word2Vec':         {'preds': preds_w2v_rf,   'probs': probs_w2v_rf,   'category': 'Классическая'},
    'RuBERT':                {'preds': preds_rubert,   'probs': probs_rubert,   'category': 'Трансформер'},
    'ruGPT-3+LoRA (V1)':     {'preds': preds_v1,       'probs': probs_v1,       'category': 'LLM (локальная)'},
    'FrozenGPT+Sk (V2)':     {'preds': preds_v2,       'probs': probs_v2,       'category': 'LLM (локальная)'},
    'ruGPT-3+LoRA (V3)':     {'preds': preds_v3,       'probs': probs_v3,       'category': 'LLM (локальная)'},
    'ruGPT-3+LoRA (V3T)':    {'preds': preds_v3t,      'probs': probs_v3t,      'category': 'LLM (локальная)'},
    'DeepSeek (API, 20-shot)': {'preds': preds_deepseek, 'probs': probs_deepseek, 'category': 'LLM (API)'},
}

rows = []
for name, d in all_models.items():
    rows.append({
        'Модель':    name,
        'Категория': d['category'],
        'Accuracy':  accuracy_score(y_true, d['preds']),
        'F1':        f1_score(y_true, d['preds'], average='weighted'),
        'Precision': precision_score(y_true, d['preds'], average='weighted'),
        'Recall':    recall_score(y_true, d['preds'], average='weighted'),
    })

results_df = pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df.index = range(1, len(results_df) + 1)
results_df.style.format({
    'Accuracy': '{:.4f}', 'F1': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}'
}).highlight_max(subset=['Accuracy', 'F1', 'Precision', 'Recall'], color='#2d5f2d')

## 8. Столбчатые диаграммы (Accuracy и F1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
df_sorted = results_df.sort_values('Accuracy', ascending=True)
colors = {
    'Классическая':    '#5B9BD5',
    'Трансформер':     '#70AD47',
    'LLM (локальная)': '#ED7D31',
    'LLM (API)':       '#9B59B6',
}
bar_colors = [colors[c] for c in df_sorted['Категория']]

axes[0].barh(df_sorted['Модель'], df_sorted['Accuracy'], color=bar_colors)
axes[0].set_xlabel('Accuracy'); axes[0].set_title('Accuracy всех моделей')
axes[0].set_xlim(0.5, 1.02)
for i, val in enumerate(df_sorted['Accuracy']):
    axes[0].text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=10)

axes[1].barh(df_sorted['Модель'], df_sorted['F1'], color=bar_colors)
axes[1].set_xlabel('F1 (weighted)'); axes[1].set_title('F1-score всех моделей')
axes[1].set_xlim(0.5, 1.02)
for i, val in enumerate(df_sorted['F1']):
    axes[1].text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=10)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
Path('assets').mkdir(exist_ok=True)
plt.savefig('assets/final_comparison_accuracy_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Все 4 метрики — группированная диаграмма

In [ ]:
metrics_cols = ['Accuracy', 'F1', 'Precision', 'Recall']
df_plot = results_df.sort_values('Accuracy', ascending=False)

fig, ax = plt.subplots(figsize=(20, 8))
x = np.arange(len(df_plot))
width = 0.2

for i, metric in enumerate(metrics_cols):
    ax.bar(x + i * width, df_plot[metric], width, label=metric)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_plot['Модель'], rotation=30, ha='right', fontsize=10)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel('Значение метрики')
ax.set_title('Сравнение всех моделей по четырём метрикам (включая DeepSeek API)')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('assets/final_comparison_4metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Матрицы ошибок всех моделей

In [ ]:
model_names = list(all_models.keys())
n_cols = 4
n_rows = (len(model_names) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, name in enumerate(model_names):
    cm = confusion_matrix(y_true, all_models[name]['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Фейк', 'Реальная'], yticklabels=['Фейк', 'Реальная'])
    axes[i].set_title(name, fontsize=11)
    axes[i].set_ylabel('Истина'); axes[i].set_xlabel('Предсказание')

for j in range(len(model_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Матрицы ошибок всех моделей', fontsize=16, y=1.005)
plt.tight_layout()
plt.savefig('assets/final_comparison_cm.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. ROC-кривые

Для DeepSeek нет калиброванных вероятностей — выводим только дискретную точку (TPR/FPR).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

line_styles = {
    'Классическая':    '--',
    'Трансформер':     '-',
    'LLM (локальная)': '-.',
    'LLM (API)':       ':',
}
cat_cmaps = {
    'Классическая':    plt.cm.Blues,
    'Трансформер':     plt.cm.Greens,
    'LLM (локальная)': plt.cm.Oranges,
    'LLM (API)':       plt.cm.Purples,
}

cat_counts = {}
for name, d in all_models.items():
    cat = d['category']
    cat_counts.setdefault(cat, 0)
    idx = cat_counts[cat]
    n_in_cat = sum(1 for v in all_models.values() if v['category'] == cat)
    cmap = cat_cmaps[cat]
    color = cmap(0.5 + 0.4 * idx / max(n_in_cat - 1, 1))
    ls = line_styles[cat]

    if cat == 'LLM (API)':
        # Для DeepSeek — дискретная точка
        from sklearn.metrics import confusion_matrix as cm_fn
        cm = cm_fn(y_true, d['preds'])
        tn, fp, fn, tp = cm.ravel()
        fpr_pt = fp / (fp + tn); tpr_pt = tp / (tp + fn)
        ax.scatter([fpr_pt], [tpr_pt], color=color, s=200, marker='*',
                   label=f'{name} (point)', zorder=5, edgecolors='black', linewidths=1)
    else:
        fpr, tpr, _ = roc_curve(y_true, d['probs'])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2,
                label=f'{name} (AUC={roc_auc:.4f})')
    cat_counts[cat] += 1

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-кривые всех моделей')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('assets/final_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Прицельное сравнение: DeepSeek API vs RuBERT vs локальные LLM

Подробный взгляд на топ-кандидатов: API-модель в zero/few-shot режиме без обучения против fine-tuned на своём датасете моделях.

In [ ]:
compare_names = ['DeepSeek (API, 20-shot)', 'RuBERT', 'ruGPT-3+LoRA (V3)', 'ruGPT-3+LoRA (V3T)']
compare_rows = []
for name in compare_names:
    p = all_models[name]['preds']
    compare_rows.append({
        'Модель':    name,
        'Accuracy':  accuracy_score(y_true, p),
        'F1':        f1_score(y_true, p, average='weighted'),
        'Precision': precision_score(y_true, p, average='weighted'),
        'Recall':    recall_score(y_true, p, average='weighted'),
    })
compare_df = pd.DataFrame(compare_rows).set_index('Модель')
print('=' * 70)
print('DeepSeek API vs лучшие локальные модели')
print('=' * 70)
print(compare_df.to_string(float_format=lambda x: f'{x:.4f}'))
print()
for name in compare_names:
    print(f'\n--- {name} ---')
    print(classification_report(y_true, all_models[name]['preds'],
                                target_names=['Фейк', 'Реальная'], digits=4))

# Confusion matrices side-by-side
fig, axes = plt.subplots(1, len(compare_names), figsize=(5 * len(compare_names), 5))
for ax, name in zip(axes, compare_names):
    cm = confusion_matrix(y_true, all_models[name]['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Фейк', 'Реальная'], yticklabels=['Фейк', 'Реальная'],
                annot_kws={'size': 14})
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Истина'); ax.set_xlabel('Предсказание')
plt.suptitle('Матрицы ошибок: DeepSeek vs топ локальные модели', fontsize=15, y=1.03)
plt.tight_layout()
plt.savefig('assets/final_deepseek_vs_top.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Сохранение результатов

In [ ]:
results_df.to_csv('assets/final_comparison.csv', index=False)
compare_df.to_csv('assets/final_deepseek_vs_top.csv')
print('Сохранено:')
for f in [
    'assets/final_comparison.csv',
    'assets/final_deepseek_vs_top.csv',
    'assets/final_comparison_accuracy_f1.png',
    'assets/final_comparison_4metrics.png',
    'assets/final_comparison_cm.png',
    'assets/final_comparison_roc.png',
    'assets/final_deepseek_vs_top.png',
    'models/deepseek/predictions_441.csv',
]:
    print(f'  {f}')

## 14. Итог

**Что показывает сравнение:**

- Все модели оценены на **одном** in-distribution hold-out (441 пример), включая DeepSeek через API.
- DeepSeek работает в режиме **20-shot** (10 фейков + 10 правды из train), без какого-либо обучения на этом датасете.
- Fine-tuned модели (RuBERT, V3, V3T) видели train — это даёт им ~98%, что близко к потолку датасета.
- DeepSeek в zero-обучения режиме показывает результат, сравнимый с дообученными моделями, **только за счёт промпт-инжиниринга**.

**Кэширование:** результаты DeepSeek сохраняются в `models/deepseek/predictions_441.csv` — повторный запуск ноутбука не требует новых вызовов API.